In [ ]:
!nvidia-smi
# !lscpu | head -n 15

In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pydicom import dcmread
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm


import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils import data

import torch.nn.functional as F
from torchvision.utils import make_grid, save_image

from matplotlib import rcParams

import matplotlib.patches as patches
from math import ceil

from dataset.dataset import TrainAndValidateDataset
from resnet import Resnet

device = torch.device('cuda:3' if torch.cuda.is_available() else 'cpu')

In [ ]:
train_and_validate_dataset = TrainAndValidateDataset()
label_data = train_and_validate_dataset.label_data
label_data.head(5)

In [ ]:
train_loader = train_and_validate_dataset.train_loader
val_loader = train_and_validate_dataset.val_loader
test_loader = train_and_validate_dataset.test_loader

# train_labels = train_and_validate_dataset.train_labels
# val_labels = train_and_validate_dataset.val_labels

# print(train_labels.shape)
# print(val_labels.shape)

# print(f'patientId: {train_labels[0][0]}, Target: {train_labels[0][1]}')

# train_paths = train_and_validate_dataset.train_paths
# val_paths = train_and_validate_dataset.val_paths

# print(len(train_paths))
# print(len(val_paths))

# train_and_validate_dataset.show_raw_dataset()

# train_and_validate_dataset.show_train_dataset()

# train_and_validate_dataset.show_dataloader()


<a id="model"></a>
# <div style="padding:20px;color:white;margin:0;font-size:20px;font-family:Georgia;text-align:left;display:fill;border-radius:5px;background-color:#254E58;overflow:hidden"><b>Loading a Pre-trained ResNet18 Model and its Fine-tuning</b></div>    

In [ ]:
model = Resnet(device=device)
criterion = nn.CrossEntropyLoss()
# Observe that all parameters are being optimized
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
# Decay LR by a factor of 0.1 every 7 epochs
exp_lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)
print(model)

<a id="train"></a>
# <div style="padding:20px;color:white;margin:0;font-size:20px;font-family:Georgia;text-align:left;display:fill;border-radius:5px;background-color:#254E58;overflow:hidden"><b>Training the Model</b></div>    

In [ ]:
num_epochs = 20
# Train the model
total_step = len(train_loader)
for epoch in range(num_epochs):
    # Training step
    model.train()
    for i, (images, labels, _) in tqdm(enumerate(train_loader)):
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i + 1) % 2000 == 0:
            print("Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}"
                    .format(epoch + 1, num_epochs, i + 1, total_step, loss.item()))
    
    exp_lr_scheduler.step()

    # Validation step
    correct = 0
    total = 0
    model.eval()
    for images, labels, _ in tqdm(val_loader):
        images = images.to(device)
        labels = labels.to(device)
        predictions = model(images)
        _, predicted = torch.max(predictions, 1)
        total += labels.size(0)
        correct += (labels == predicted).sum()
    print(f'Epoch: {epoch + 1}/{num_epochs}, Val_Acc: {100 * correct / total}')

<a id="test"></a>
# <div style="padding:20px;color:white;margin:0;font-size:20px;font-family:Georgia;text-align:left;display:fill;border-radius:5px;background-color:#254E58;overflow:hidden"><b>Testing the Model</b></div>    

In [ ]:
model.eval()

correct = 0
total = 0
for images, labels, _ in tqdm(test_loader):
    images = images.to(device)
    labels = labels.to(device)
    predictions = model(images)
    _, predicted = torch.max(predictions, 1)
    total += labels.size(0)
    correct += (labels == predicted).sum()
print(f'Test_Acc: {100 * correct / total}')

<a id="save"></a>
# <div style="padding:20px;color:white;margin:0;font-size:20px;font-family:Georgia;text-align:left;display:fill;border-radius:5px;background-color:#254E58;overflow:hidden"><b>Save the Model Weights</b></div>      

In [ ]:
# torch.save(model.state_dict(), './rsna-dataset/my_models_weights.pth')
torch.save(model, './rsna-dataset/model_10092024.pth')
print("Model and weights saved.")

<a id="load"></a>
# <div style="padding:20px;color:white;margin:0;font-size:20px;font-family:Georgia;text-align:left;display:fill;border-radius:5px;background-color:#254E58;overflow:hidden"><b>Loading Saved Model</b></div>     

In [ ]:
model = torch.load('./rsna-dataset/my_model.pth')

In [ ]:
model.to(device)
model.eval()

correct = 0
total = 0  
for images, labels, _ in tqdm(test_loader):
    images = images.to(device)
    labels = labels.to(device)
    predictions = model(images)
    _, predicted = torch.max(predictions, 1)
    total += labels.size(0)
    correct += (labels == predicted).sum()
print(f'Test_Acc: {100*correct/total}')

In [ ]:
# Load test image
label = 0
while(label==0):
    pil_img, label, box = next(image)

fig,ax = plt.subplots(1)

Orig_img_size = 1024
img_size = 224

# 'r' means relative. 'c' means center.
rx = ceil(box[0]*img_size/Orig_img_size)
ry = ceil(box[1]*img_size/Orig_img_size)
rw = ceil(box[2]*img_size/Orig_img_size)
rh = ceil(box[3]*img_size/Orig_img_size)


pil_img = np.transpose(pil_img, (1, 2, 0))
print(pil_img.shape)
rect = patches.Rectangle((rx, ry), rw, rh, linewidth=1, edgecolor='r', facecolor='none')
ax.imshow(pil_img)
ax.add_patch(rect)
print("Label : ", label, box)